In [1]:
!git clone -b megha-model-extension https://github.com/meghagangal362015/cmpe257-early-reading-prediction.git

%cd cmpe257-early-reading-prediction


Cloning into 'cmpe257-early-reading-prediction'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 120 (delta 44), reused 85 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 1.51 MiB | 4.83 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/cmpe257-early-reading-prediction


In [2]:
import pandas as pd

paper_df = pd.read_csv("data/processed/paper_complete_706.csv")

print("Dataset shape:", paper_df.shape)
print("Missing values:", paper_df.isna().sum().sum())

Dataset shape: (706, 19)
Missing values: 0


In [3]:
# Define the 16 paper features and target columns

paper_features = [
    "NWFcls",
    "NWFwrc",
    "ORFwc",
    "SAwrS",
    "SAsrS",
    "SAtoS",
    "RMwidRS",
    "RMwdaRS",
    "Gender",
    "Tx",
    "Age1b",
    "Tier2_N",
    "TKPctCrct",
    "grp_rate",
    "rcmistot",
    "gnrl_fid"
]

X = paper_df[paper_features].copy()
groups = paper_df["SchlID"]

targets = {
    "Word Identification": "RMwidRS_target",
    "Word Attack": "RMwdaRS_target"
}

print("Features:", X.shape)
print("Schools:", groups.nunique())

Features: (706, 16)
Schools: 42


In [4]:
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, average_precision_score
from sklearn.base import clone
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]),
        numeric_features
    )
])

group_kfold = GroupKFold(n_splits=5)

print("Preprocessing pipeline ready.")
print("Five school-grouped folds ready.")

Preprocessing pipeline ready.
Five school-grouped folds ready.


In [6]:
rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10],
    "min_samples_leaf": [1, 2, 5]
}

rf_tuning_results = []

for target_name, target_column in targets.items():

    y = paper_df[target_column].astype(int)

    for params in ParameterGrid(rf_param_grid):

        fold_scores = []
        fold_accuracies = []

        print(
            f"{target_name} | "
            f"n_estimators={params['n_estimators']}, "
            f"max_depth={params['max_depth']}, "
            f"min_samples_leaf={params['min_samples_leaf']}"
        )

        for train_index, validation_index in group_kfold.split(
            X, y, groups=groups
        ):
            X_train = X.iloc[train_index]
            X_validation = X.iloc[validation_index]

            y_train = y.iloc[train_index]
            y_validation = y.iloc[validation_index]

            model = Pipeline([
                ("preprocessor", clone(preprocessor)),
                (
                    "classifier",
                    RandomForestClassifier(
                        **params,
                        random_state=42,
                        class_weight="balanced",
                        n_jobs=-1
                    )
                )
            ])

            model.fit(X_train, y_train)

            probabilities = model.predict_proba(
                X_validation
            )[:, 1]

            predictions = model.predict(X_validation)

            fold_scores.append(
                average_precision_score(
                    y_validation,
                    probabilities
                )
            )

            fold_accuracies.append(
                accuracy_score(
                    y_validation,
                    predictions
                )
            )

        rf_tuning_results.append({
            "Target": target_name,
            **params,
            "Mean PR-AUC": np.mean(fold_scores),
            "PR-AUC SD": np.std(fold_scores, ddof=1),
            "Mean Accuracy": np.mean(fold_accuracies),
            "Accuracy SD": np.std(fold_accuracies, ddof=1)
        })

rf_tuning_df = pd.DataFrame(rf_tuning_results)

Word Identification | n_estimators=200, max_depth=None, min_samples_leaf=1
Word Identification | n_estimators=400, max_depth=None, min_samples_leaf=1
Word Identification | n_estimators=200, max_depth=None, min_samples_leaf=2
Word Identification | n_estimators=400, max_depth=None, min_samples_leaf=2
Word Identification | n_estimators=200, max_depth=None, min_samples_leaf=5
Word Identification | n_estimators=400, max_depth=None, min_samples_leaf=5
Word Identification | n_estimators=200, max_depth=10, min_samples_leaf=1
Word Identification | n_estimators=400, max_depth=10, min_samples_leaf=1
Word Identification | n_estimators=200, max_depth=10, min_samples_leaf=2
Word Identification | n_estimators=400, max_depth=10, min_samples_leaf=2
Word Identification | n_estimators=200, max_depth=10, min_samples_leaf=5
Word Identification | n_estimators=400, max_depth=10, min_samples_leaf=5
Word Attack | n_estimators=200, max_depth=None, min_samples_leaf=1
Word Attack | n_estimators=400, max_depth=Non

In [7]:
rf_tuning_df = rf_tuning_df.sort_values(
    ["Target", "Mean PR-AUC"],
    ascending=[True, False]
)

best_rf_models = (
    rf_tuning_df
    .groupby("Target", as_index=False)
    .first()
)

print("Best Random Forest configuration for each target:")
display(best_rf_models)

Best Random Forest configuration for each target:


,Target,max_depth,min_samples_leaf,n_estimators,Mean PR-AUC,PR-AUC SD,Mean Accuracy,Accuracy SD
0,Word Attack,10.0,2,400,0.677675,0.097843,0.584849,0.057293
1,Word Identification,10.0,1,400,0.693540,0.100208,0.634708,0.053259


In [8]:
xgb_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [2, 3],
    "learning_rate": [0.03, 0.05]
}

xgb_tuning_results = []

for target_name, target_column in targets.items():

    y = paper_df[target_column].astype(int)

    for params in ParameterGrid(xgb_param_grid):

        fold_scores = []
        fold_accuracies = []

        print(
            f"{target_name} | "
            f"n_estimators={params['n_estimators']}, "
            f"max_depth={params['max_depth']}, "
            f"learning_rate={params['learning_rate']}"
        )

        for train_index, validation_index in group_kfold.split(
            X, y, groups=groups
        ):
            X_train = X.iloc[train_index]
            X_validation = X.iloc[validation_index]

            y_train = y.iloc[train_index]
            y_validation = y.iloc[validation_index]

            model = Pipeline([
                ("preprocessor", clone(preprocessor)),
                (
                    "classifier",
                    XGBClassifier(
                        **params,
                        subsample=0.8,
                        colsample_bytree=0.8,
                        objective="binary:logistic",
                        eval_metric="logloss",
                        random_state=42,
                        n_jobs=-1
                    )
                )
            ])

            model.fit(X_train, y_train)

            probabilities = model.predict_proba(
                X_validation
            )[:, 1]

            predictions = model.predict(X_validation)

            fold_scores.append(
                average_precision_score(
                    y_validation,
                    probabilities
                )
            )

            fold_accuracies.append(
                accuracy_score(
                    y_validation,
                    predictions
                )
            )

        xgb_tuning_results.append({
            "Target": target_name,
            **params,
            "Mean PR-AUC": np.mean(fold_scores),
            "PR-AUC SD": np.std(fold_scores, ddof=1),
            "Mean Accuracy": np.mean(fold_accuracies),
            "Accuracy SD": np.std(fold_accuracies, ddof=1)
        })

xgb_tuning_df = pd.DataFrame(xgb_tuning_results)

Word Identification | n_estimators=200, max_depth=2, learning_rate=0.03
Word Identification | n_estimators=400, max_depth=2, learning_rate=0.03
Word Identification | n_estimators=200, max_depth=3, learning_rate=0.03
Word Identification | n_estimators=400, max_depth=3, learning_rate=0.03
Word Identification | n_estimators=200, max_depth=2, learning_rate=0.05
Word Identification | n_estimators=400, max_depth=2, learning_rate=0.05
Word Identification | n_estimators=200, max_depth=3, learning_rate=0.05
Word Identification | n_estimators=400, max_depth=3, learning_rate=0.05
Word Attack | n_estimators=200, max_depth=2, learning_rate=0.03
Word Attack | n_estimators=400, max_depth=2, learning_rate=0.03
Word Attack | n_estimators=200, max_depth=3, learning_rate=0.03
Word Attack | n_estimators=400, max_depth=3, learning_rate=0.03
Word Attack | n_estimators=200, max_depth=2, learning_rate=0.05
Word Attack | n_estimators=400, max_depth=2, learning_rate=0.05
Word Attack | n_estimators=200, max_dept

In [9]:
xgb_tuning_df = xgb_tuning_df.sort_values(
    ["Target", "Mean PR-AUC"],
    ascending=[True, False]
)

best_xgb_models = (
    xgb_tuning_df
    .groupby("Target", as_index=False)
    .first()
)

print("Best XGBoost configuration for each target:")
display(best_xgb_models)

Best XGBoost configuration for each target:


,Target,learning_rate,max_depth,n_estimators,Mean PR-AUC,PR-AUC SD,Mean Accuracy,Accuracy SD
0,Word Attack,0.05,2,400,0.686303,0.115645,0.611831,0.055027
1,Word Identification,0.03,2,200,0.711751,0.097436,0.655835,0.055446


In [10]:
best_rf_comparison = best_rf_models.copy()
best_rf_comparison["Model"] = "Tuned Random Forest"

best_xgb_comparison = best_xgb_models.copy()
best_xgb_comparison["Model"] = "Tuned XGBoost"

tuned_comparison = pd.concat(
    [best_rf_comparison, best_xgb_comparison],
    ignore_index=True
)

tuned_comparison = tuned_comparison[
    [
        "Target",
        "Model",
        "Mean Accuracy",
        "Accuracy SD",
        "Mean PR-AUC",
        "PR-AUC SD"
    ]
]

display(tuned_comparison)

,Target,Model,Mean Accuracy,Accuracy SD,Mean PR-AUC,PR-AUC SD
0,Word Attack,Tuned Random Forest,0.584849,0.057293,0.677675,0.097843
1,Word Identification,Tuned Random Forest,0.634708,0.053259,0.693540,0.100208
2,Word Attack,Tuned XGBoost,0.611831,0.055027,0.686303,0.115645
3,Word Identification,Tuned XGBoost,0.655835,0.055446,0.711751,0.097436


In [11]:
feature_groups = {
    "Reading assessment features": [
        "NWFcls",
        "NWFwrc",
        "ORFwc",
        "SAwrS",
        "SAsrS",
        "SAtoS",
        "RMwidRS",
        "RMwdaRS"
    ],
    "Student features": [
        "Gender",
        "Tx",
        "Age1b"
    ],
    "Classroom features": [
        "Tier2_N",
        "TKPctCrct",
        "grp_rate",
        "rcmistot",
        "gnrl_fid"
    ]
}

for group_name, features in feature_groups.items():
    print(group_name, ":", len(features), "features")

Reading assessment features : 8 features
Student features : 3 features
Classroom features : 5 features


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

def clean_depth(value):
    if pd.isna(value):
        return None
    return int(value)


ablation_results = []

all_ablation_groups = {
    "None removed": [],
    **feature_groups
}

for target_name, target_column in targets.items():

    y = paper_df[target_column].astype(int)

    rf_params = best_rf_models[
        best_rf_models["Target"] == target_name
    ].iloc[0]

    xgb_params = best_xgb_models[
        best_xgb_models["Target"] == target_name
    ].iloc[0]

    model_configs = {
        "Tuned Random Forest": {
            "n_estimators": int(rf_params["n_estimators"]),
            "max_depth": clean_depth(rf_params["max_depth"]),
            "min_samples_leaf": int(rf_params["min_samples_leaf"])
        },
        "Tuned XGBoost": {
            "n_estimators": int(xgb_params["n_estimators"]),
            "max_depth": clean_depth(xgb_params["max_depth"]),
            "learning_rate": float(xgb_params["learning_rate"]),
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "objective": "binary:logistic",
            "eval_metric": "logloss"
        }
    }

    for model_name, model_params in model_configs.items():

        for removed_group, removed_features in all_ablation_groups.items():

            selected_features = [
                feature for feature in paper_features
                if feature not in removed_features
            ]

            X_ablation = paper_df[selected_features].copy()

            ablation_preprocessor = ColumnTransformer([
                (
                    "numeric",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median"))
                    ]),
                    selected_features
                )
            ])

            fold_pr_auc = []
            fold_accuracy = []

            for train_index, validation_index in group_kfold.split(
                X_ablation,
                y,
                groups=groups
            ):
                X_train = X_ablation.iloc[train_index]
                X_validation = X_ablation.iloc[validation_index]

                y_train = y.iloc[train_index]
                y_validation = y.iloc[validation_index]

                if model_name == "Tuned Random Forest":
                    classifier = RandomForestClassifier(
                        **model_params,
                        random_state=42,
                        class_weight="balanced",
                        n_jobs=-1
                    )
                else:
                    classifier = XGBClassifier(
                        **model_params,
                        random_state=42,
                        n_jobs=-1
                    )

                model = Pipeline([
                    ("preprocessor", ablation_preprocessor),
                    ("classifier", classifier)
                ])

                model.fit(X_train, y_train)

                probabilities = model.predict_proba(
                    X_validation
                )[:, 1]

                predictions = model.predict(X_validation)

                fold_pr_auc.append(
                    average_precision_score(
                        y_validation,
                        probabilities
                    )
                )

                fold_accuracy.append(
                    accuracy_score(
                        y_validation,
                        predictions
                    )
                )

            ablation_results.append({
                "Target": target_name,
                "Model": model_name,
                "Removed group": removed_group,
                "Mean Accuracy": np.mean(fold_accuracy),
                "Mean PR-AUC": np.mean(fold_pr_auc),
                "PR-AUC SD": np.std(fold_pr_auc, ddof=1)
            })

ablation_df = pd.DataFrame(ablation_results)

display(ablation_df.round(4))

,Target,Model,Removed group,Mean Accuracy,Mean PR-AUC,PR-AUC SD
0,Word Identification,Tuned Random Forest,None removed,0.6347,0.6935,0.1002
1,Word Identification,Tuned Random Forest,Reading assessment features,0.5172,0.5802,0.0538
2,Word Identification,Tuned Random Forest,Student features,0.6247,0.6863,0.1123
3,Word Identification,Tuned Random Forest,Classroom features,0.6375,0.7015,0.0766
4,Word Identification,Tuned XGBoost,None removed,0.6558,0.7118,0.0974
5,Word Identification,Tuned XGBoost,Reading assessment features,0.5172,0.5861,0.0379
6,Word Identification,Tuned XGBoost,Student features,0.6474,0.7059,0.1070
7,Word Identification,Tuned XGBoost,Classroom features,0.6530,0.6881,0.0878
8,Word Attack,Tuned Random Forest,None removed,0.5933,0.6771,0.0952
9,Word Attack,Tuned Random Forest,Reading assessment features,0.5667,0.6409,0.0906


In [16]:
# Sort ablation results by target, model, and PR-AUC

ablation_summary = (
    ablation_df
    .sort_values(
        ["Target", "Model", "Mean PR-AUC"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

display(ablation_summary.round(4))

,Target,Model,Removed group,Mean Accuracy,Mean PR-AUC,PR-AUC SD
0,Word Attack,Tuned Random Forest,Classroom features,0.5921,0.6875,0.0998
1,Word Attack,Tuned Random Forest,None removed,0.5933,0.6771,0.0952
2,Word Attack,Tuned Random Forest,Student features,0.5933,0.6736,0.0943
3,Word Attack,Tuned Random Forest,Reading assessment features,0.5667,0.6409,0.0906
4,Word Attack,Tuned XGBoost,Classroom features,0.6063,0.6991,0.0938
5,Word Attack,Tuned XGBoost,None removed,0.6118,0.6863,0.1156
6,Word Attack,Tuned XGBoost,Student features,0.6076,0.6757,0.0939
7,Word Attack,Tuned XGBoost,Reading assessment features,0.5737,0.6276,0.0885
8,Word Identification,Tuned Random Forest,Classroom features,0.6375,0.7015,0.0766
9,Word Identification,Tuned Random Forest,None removed,0.6347,0.6935,0.1002


In [17]:
baseline_scores = (
    ablation_df[
        ablation_df["Removed group"] == "None removed"
    ][
        ["Target", "Model", "Mean PR-AUC"]
    ]
    .rename(columns={"Mean PR-AUC": "Baseline PR-AUC"})
)

ablation_with_baseline = ablation_df.merge(
    baseline_scores,
    on=["Target", "Model"],
    how="left"
)

ablation_with_baseline["PR-AUC Change"] = (
    ablation_with_baseline["Mean PR-AUC"]
    - ablation_with_baseline["Baseline PR-AUC"]
)

ablation_with_baseline = ablation_with_baseline.sort_values(
    ["Target", "Model", "PR-AUC Change"]
)

display(ablation_with_baseline.round(4))

,Target,Model,Removed group,Mean Accuracy,Mean PR-AUC,PR-AUC SD,Baseline PR-AUC,PR-AUC Change
9,Word Attack,Tuned Random Forest,Reading assessment features,0.5667,0.6409,0.0906,0.6771,-0.0362
10,Word Attack,Tuned Random Forest,Student features,0.5933,0.6736,0.0943,0.6771,-0.0035
8,Word Attack,Tuned Random Forest,None removed,0.5933,0.6771,0.0952,0.6771,0.0000
11,Word Attack,Tuned Random Forest,Classroom features,0.5921,0.6875,0.0998,0.6771,0.0104
13,Word Attack,Tuned XGBoost,Reading assessment features,0.5737,0.6276,0.0885,0.6863,-0.0587
14,Word Attack,Tuned XGBoost,Student features,0.6076,0.6757,0.0939,0.6863,-0.0106
12,Word Attack,Tuned XGBoost,None removed,0.6118,0.6863,0.1156,0.6863,0.0000
15,Word Attack,Tuned XGBoost,Classroom features,0.6063,0.6991,0.0938,0.6863,0.0128
1,Word Identification,Tuned Random Forest,Reading assessment features,0.5172,0.5802,0.0538,0.6935,-0.1134
2,Word Identification,Tuned Random Forest,Student features,0.6247,0.6863,0.1123,0.6935,-0.0072


In [18]:
import os

os.makedirs("results", exist_ok=True)

rf_tuning_df.to_csv(
    "results/random_forest_tuning_results.csv",
    index=False
)

xgb_tuning_df.to_csv(
    "results/xgboost_tuning_results.csv",
    index=False
)

tuned_comparison.to_csv(
    "results/tuned_model_comparison.csv",
    index=False
)

ablation_with_baseline.to_csv(
    "results/feature_group_ablation_results.csv",
    index=False
)

print("Tuning and ablation results saved.")

Tuning and ablation results saved.


In [19]:
print("Best Random Forest models:")
display(best_rf_models.round(4))

print("Best XGBoost models:")
display(best_xgb_models.round(4))

print("Tuned model comparison:")
display(tuned_comparison.round(4))

print("Feature-group ablation:")
display(ablation_with_baseline.round(4))

Best Random Forest models:


,Target,max_depth,min_samples_leaf,n_estimators,Mean PR-AUC,PR-AUC SD,Mean Accuracy,Accuracy SD
0,Word Attack,10.0,2,400,0.6777,0.0978,0.5848,0.0573
1,Word Identification,10.0,1,400,0.6935,0.1002,0.6347,0.0533


Best XGBoost models:


,Target,learning_rate,max_depth,n_estimators,Mean PR-AUC,PR-AUC SD,Mean Accuracy,Accuracy SD
0,Word Attack,0.05,2,400,0.6863,0.1156,0.6118,0.0550
1,Word Identification,0.03,2,200,0.7118,0.0974,0.6558,0.0554


Tuned model comparison:


,Target,Model,Mean Accuracy,Accuracy SD,Mean PR-AUC,PR-AUC SD
0,Word Attack,Tuned Random Forest,0.5848,0.0573,0.6777,0.0978
1,Word Identification,Tuned Random Forest,0.6347,0.0533,0.6935,0.1002
2,Word Attack,Tuned XGBoost,0.6118,0.0550,0.6863,0.1156
3,Word Identification,Tuned XGBoost,0.6558,0.0554,0.7118,0.0974


Feature-group ablation:


,Target,Model,Removed group,Mean Accuracy,Mean PR-AUC,PR-AUC SD,Baseline PR-AUC,PR-AUC Change
9,Word Attack,Tuned Random Forest,Reading assessment features,0.5667,0.6409,0.0906,0.6771,-0.0362
10,Word Attack,Tuned Random Forest,Student features,0.5933,0.6736,0.0943,0.6771,-0.0035
8,Word Attack,Tuned Random Forest,None removed,0.5933,0.6771,0.0952,0.6771,0.0000
11,Word Attack,Tuned Random Forest,Classroom features,0.5921,0.6875,0.0998,0.6771,0.0104
13,Word Attack,Tuned XGBoost,Reading assessment features,0.5737,0.6276,0.0885,0.6863,-0.0587
14,Word Attack,Tuned XGBoost,Student features,0.6076,0.6757,0.0939,0.6863,-0.0106
12,Word Attack,Tuned XGBoost,None removed,0.6118,0.6863,0.1156,0.6863,0.0000
15,Word Attack,Tuned XGBoost,Classroom features,0.6063,0.6991,0.0938,0.6863,0.0128
1,Word Identification,Tuned Random Forest,Reading assessment features,0.5172,0.5802,0.0538,0.6935,-0.1134
2,Word Identification,Tuned Random Forest,Student features,0.6247,0.6863,0.1123,0.6935,-0.0072
